# Set Configuration

In [1]:
from diffusion_hash_inv.config import MainConfig, HashConfig, MessageConfig, OutputConfig, Byte2RGBConfig
from diffusion_hash_inv.main import MainEP
from diffusion_hash_inv.utils import FileIO
from diffusion_hash_inv.main import RuntimeConfig

length = 256
iteration = 10_000
RUN = True

main_cfg = MainConfig(
    verbose_flag=False,
    clean_flag=False,
    debug_flag=False,
    make_image_flag=True,
)
hash_cfg = HashConfig(
    hash_alg="md5",
    length=length,
)
message_cfg = MessageConfig(
    message_flag=False,
    length=length,
    random_flag=True,
    seed_flag=True,
)
output_cfg = OutputConfig()
byte2rgb_cfg = Byte2RGBConfig()
runtime_cfg = RuntimeConfig(
    main=main_cfg,
    message=message_cfg,
    hash=hash_cfg,
    output=output_cfg,
    rgb=byte2rgb_cfg,
)

io_controller = FileIO(main_config=main_cfg, output_cfg=output_cfg)

In [ ]:
if RUN:
    main = MainEP(runtime_config=runtime_cfg, file_controller=io_controller)
    main.run(iteration=iteration)

Main Entry Point Initialized.
Program Start Time: 2026-05-09 01:34:57.932808+09:00
Hash Algorithm: MD5
Message Length: 256
Data Directory: /Users/choisoonwook/Experiments_local/Diffusion_HASH_inverse/data
Output Directory: /Users/choisoonwook/Experiments_local/Diffusion_HASH_inverse/output
Hash Generation Progress: 0/10000 iteration
Hash Generation Progress: 5695/10000 iteration Hash Algorithm=MD5 Message Length=256
Hash Generation Progress: 10000/10000 iteration Hash Algorithm=MD5 Message Length=256
Hash Generation Progress: done 10000/10000 iteration Hash Algorithm=MD5 Message Length=256
Hash Calculation time: 8 s, 675 ms, 986 us, 750 ns
Process completed.

RGB Image Maker Module Loaded.
RGB Image Maker Initialized.
Found 10000 logs to process.
Expected 710000 images to write (71 images per log).
Processing Logs: 0/10000 log
Writing Images: 0/710000 image
Writing Images: 6047/710000 image
Processing Logs: 86/10000 log images=6035
Writing Images: 12137/710000 image
Processing Logs: 17

In [ ]:
from diffusion_hash_inv.logger.logger import Logs

logs = Logs.get_logs(io_controller, hash_cfg, main_cfg)
print(len(logs))
print(logs[0])

In [ ]:
def get_step4(logs):
    """
    Extract the "4th Step" logs from the given list of log dictionaries.
    """
    _step4_logs = []
    for _log in logs:
        _tmp = list(_log.values())
        assert len(_tmp) == 1, "Each log dictionary should contain exactly one key-value pair."
        log_dict = list(_log.values())[0]
        if "Logs" in log_dict and "4th Step" in log_dict["Logs"]:
            _step4_logs.append(log_dict["Logs"]["4th Step"])
    return _step4_logs

# Beta Scheduling
## Approach 1
**method**  
1. Cumulative all bytes from Hash algorithm calculation logs  
2. Make new Beta Schedule by multiply Base Beta Schedule and Cumulative block


In [ ]:
from typing import List, Dict, Any

step4_logs: List[Dict[str, Any]] = get_step4(logs)
assert len(step4_logs) == len(logs), f"Step 4 logs mismatch: {len(step4_logs)} != {len(logs)}"
print(step4_logs[0])
sn = list()

In [ ]:
def cumulative_block(byte_list: List[bytes], block: bytes, indent: int = 0) -> List[bytes]:
    """
    Seperate Block into bytes and cumulatively add to byte_list. Return the updated byte_list.
    """
    _byte = 0
    for byte in block:
        _byte = byte if len(byte_list) == 0 else byte + byte_list[-1]
        byte_list.append(_byte)
    return byte_list

def make_beta_schedule(step4_logs: Dict[str, Any]) -> List[float]:
    step4_log: Dict[str, Any] = list(step4_logs.values())
    beta_schedule = []
    for log in step4_log:
        for key, value in log.items():
            if main_cfg.verbose_flag:
                # print(f"Key: {key}")
                pass
            for k, v in value.items():
                _v = Logs.str_to_bytes(v)
                cumulative_block(beta_schedule, _v, indent=3)

    return beta_schedule

In [ ]:
from diffusion_hash_inv.utils.progress import progress

sn = []
with progress(step4_logs, total=len(step4_logs), desc="Processing Step 4 Logs", unit="log") as step4_log_process:
    for log in step4_log_process:
        sn.append(make_beta_schedule(log))


In [ ]:
print(len(sn))
print(len(sn[0]))
print(sn[0])
print(sn[1])

In [ ]:
if main_cfg.verbose_flag:
    width = len(str(iteration))
    for i, beta in enumerate(sn):
        print(f"Beta Schedule {i:0{width}}:")
        print(f"{'\t' * 1}Beta: ", end="")
        for j, b in enumerate(beta):
            print(f"{b:03},", end=" ")
        print("\n")

In [ ]:
import mlx.core as mx
import numpy as np
betas = mx.linspace(start=1e-4, stop=2e-2, num=len(sn[0]))
betas = np.array(betas, dtype=np.float64)

In [ ]:
sn_array = np.array(sn, dtype=np.uint64)
print(sn_array.shape)
_min = np.min(sn_array, axis=0)
_max = np.max(sn_array, axis=0)
mean = np.mean(sn_array, axis=0)

mean = np.array(mean, dtype=np.float64)
var = np.var(sn_array, axis=0)

var = np.array(var, dtype=np.float64)
std = np.std(sn_array, axis=0)

std = np.array(std, dtype=np.float64)

candidate_betas = np.multiply(mean, betas)
beta_candidate_from_multiply = np.array(candidate_betas, dtype=np.float64)
print(beta_candidate_from_multiply)

np.set_printoptions(threshold=np.inf, linewidth=np.inf)

sn_min = np.min(mean, axis=0)
sn_max = np.max(mean, axis=0)
ori_beta_min = np.min(betas)
ori_beta_max = np.max(betas)

print(f"Min: {_min}, Length: {len(_min)}")
print(f"Max: {_max}, Length: {len(_max)}")
print(f"Mean: {mean}, Length: {len(mean)}")
print(f"Variance: {var}, Length: {len(var)}")
print(f"Standard Deviation: {std}, Length: {len(std)}")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(20, 10))
plt.plot(_min, label='Min', color='blue')
plt.legend()
plt.title('Min Values of SN')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(_max, label='Max', color='orange')
plt.legend()
plt.title('Max Values of SN')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(mean, label='Mean', color='green')
plt.legend()
plt.title('Mean Values of SN')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(var, label='Variance', color='red')
plt.legend()
plt.title('Variance Values of SN')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(std, label='Standard Deviation', color='purple')
plt.legend()
plt.title('Standard Deviation Values of SN')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(betas, label='Betas', color='blue')
plt.legend()
plt.title('Original Betas')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(sn_array[0], label='SN from log 0', color='cyan')
plt.legend()
plt.title('SN from log 0 Values over Loops')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(beta_candidate_from_multiply, label='Candidate Betas', color='cyan')
plt.legend()
plt.title('Candidate Betas from Multiplication')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()

In [ ]:
raw_candiate_betas = np.multiply(mean, betas)
raw_min = np.min(raw_candiate_betas, axis=0)
raw_max = np.max(raw_candiate_betas, axis=0)
beta_candidate_rescaled = (raw_candiate_betas - raw_min) / (raw_max - raw_min) * (ori_beta_max - ori_beta_min) + ori_beta_min
beta_candidate_rescaled = np.clip(beta_candidate_rescaled, ori_beta_min, ori_beta_max)
beta_candidate_rescaled = np.array(beta_candidate_rescaled, dtype=np.float64)
print(beta_candidate_rescaled)

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(beta_candidate_rescaled, label='Candidate Betas', color='cyan')
plt.legend()
plt.title('Candidate Betas from Multiplication (Rescaled)')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()

# Beta Scheduling
## Approach 2
**method**  
1. Cumulative all bytes from Hash algorithm calculation logs  
2. Make new Beta scheduling with the cumulative sum of log bytes from intermediate computations on the x-axis  


In [ ]:
print(f"Original Beta Min: {ori_beta_min}")
print(f"Original Beta Max: {ori_beta_max}")
print(f"SN Min: {sn_min}")
print(f"SN Max: {sn_max}")
linear_eq_slope = (ori_beta_max - ori_beta_min) / (sn_max - sn_min)
print(f"Linear Equation Slope: {linear_eq_slope}")

In [ ]:
linear_equation = lambda x: linear_eq_slope * (x - sn_min) + ori_beta_min
beta_candidate_from_linear_eq = linear_equation(mean)
beta_candidate_from_linear_eq = np.array(beta_candidate_from_linear_eq, dtype=np.float64)
print(beta_candidate_from_linear_eq)

In [ ]:
for i in range(len(beta_candidate_from_linear_eq)):
    print(betas[i] - beta_candidate_from_linear_eq[i])

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(beta_candidate_from_linear_eq, label='Make beta schedule using linear equation', color='cyan')
plt.legend()
plt.title('Candidate Betas from Linear Equation')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()